# File Operations: Direct vs Gateway - Protocol Comparison

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.4

def load_metrics(path):
    df = pd.read_json(path, lines=True)
    pts = df[df["type"] == "Point"].copy()
    exp = pd.json_normalize(pts["data"])
    exp["metric"] = pts["metric"].values
    exp["time"] = pd.to_datetime(exp["time"], utc=True)
    exp = exp.sort_values("time").reset_index(drop=True)
    exp["time_sec"] = (exp["time"] - exp["time"].min()).dt.total_seconds()
    return exp

## Configuration

Replace `TIMESTAMP` with the folder names from `../results/`.

In [ ]:
runs = {
    "direct / http":   load_metrics("../results/storage_direct_files/http/smoke/TIMESTAMP/metrics.json"),
    "direct / https":  load_metrics("../results/storage_direct_files/https/smoke/TIMESTAMP/metrics.json"),
    "direct / mtls":   load_metrics("../results/storage_direct_files/mtls/smoke/TIMESTAMP/metrics.json"),
    "gateway / http":  load_metrics("../results/gateway_storage_files/http/smoke/TIMESTAMP/metrics.json"),
    "gateway / https": load_metrics("../results/gateway_storage_files/https/smoke/TIMESTAMP/metrics.json"),
    "gateway / mtls":  load_metrics("../results/gateway_storage_files/mtls/smoke/TIMESTAMP/metrics.json"),
}

## Stats Summary

In [ ]:
rows = []
for label, df in runs.items():
    dur = df[df["metric"] == "http_req_duration"]["value"].dropna()
    rows.append({
        "run": label,
        "count": len(dur),
        "mean": round(dur.mean(), 2),
        "p50": round(np.percentile(dur, 50), 2),
        "p90": round(np.percentile(dur, 90), 2),
        "p95": round(np.percentile(dur, 95), 2),
        "p99": round(np.percentile(dur, 99), 2),
        "max": round(dur.max(), 2),
    })
pd.DataFrame(rows).set_index("run")

## Per-Operation Stats

In [ ]:
op_rows = []
for label, df in runs.items():
    dur = df[df["metric"] == "http_req_duration"].copy()
    if "tags.operation" not in dur.columns:
        continue
    for op, grp in dur.groupby("tags.operation"):
        vals = grp["value"].dropna()
        op_rows.append({
            "run": label,
            "operation": op,
            "count": len(vals),
            "mean": round(vals.mean(), 2),
            "p50": round(np.percentile(vals, 50), 2),
            "p90": round(np.percentile(vals, 90), 2),
            "p95": round(np.percentile(vals, 95), 2),
        })

op_df = pd.DataFrame(op_rows)
op_df.set_index(["run", "operation"])

## Per-Operation Percentile Chart

In [ ]:
operations = [op for op in ["upload", "get_file", "list"] if op in op_df["operation"].values]

fig, axes = plt.subplots(1, len(operations), figsize=(6 * len(operations), 5), squeeze=False)
fig.suptitle("p95 Response Time per Operation: Direct vs Gateway (ms)", fontsize=13)

for ax, op in zip(axes[0], operations):
    sub = op_df[op_df["operation"] == op].set_index("run")[["p50", "p95"]]
    sub.plot(kind="bar", ax=ax, rot=25)
    ax.set_title(op)
    ax.set_ylabel("Duration (ms)")
    ax.set_xlabel("")
    for container in ax.containers:
        ax.bar_label(container, fmt="%.1f", fontsize=7, padding=2)

plt.tight_layout()
plt.show()

## Response Time Over Time

In [ ]:
colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#1f77b4", "#ff7f0e", "#2ca02c"]
styles = ["-", "-", "-", "--", "--", "--"]

for op in operations:
    fig, ax = plt.subplots(figsize=(12, 5))
    for (label, df), color, style in zip(runs.items(), colors, styles):
        dur = df[df["metric"] == "http_req_duration"].copy()
        if "tags.operation" not in dur.columns:
            continue
        grp = dur[dur["tags.operation"] == op].set_index("time_sec")["value"]
        if grp.empty:
            continue
        smoothed = grp.rolling(window=10, min_periods=1).mean()
        ax.plot(smoothed.index, smoothed.values, label=label, color=color, linestyle=style, linewidth=1.3, alpha=0.9)
    ax.set_title(f"{op} response time (ms, smoothed) - solid: direct, dashed: gateway")
    ax.set_xlabel("Elapsed time (s)")
    ax.set_ylabel("Duration (ms)")
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

## TLS / Connection Timing Breakdown

In [ ]:
timing_metrics = [
    "http_req_blocked",
    "http_req_connecting",
    "http_req_tls_handshaking",
    "http_req_sending",
    "http_req_waiting",
    "http_req_receiving",
]
colors = ["#e07b39", "#5b9bd5", "#70ad47", "#ffc000", "#4472c4", "#ed7d31"]

timing_data = {}
for label, df in runs.items():
    timing_data[label] = {
        m: df[df["metric"] == m]["value"].dropna().mean() or 0
        for m in timing_metrics
    }

pd.DataFrame(timing_data).T.plot(
    kind="bar", stacked=True, figsize=(11, 5), color=colors, rot=25
)
plt.title("Mean Request Timing Breakdown (ms)")
plt.ylabel("Mean Duration (ms)")
plt.legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.show()

## Check Pass Rate

In [ ]:
check_rows = {}
for label, df in runs.items():
    checks = df[df["metric"] == "checks"]
    if checks.empty:
        continue
    total = len(checks)
    passed = (checks["value"] == 1).sum()
    check_rows[label] = {
        "passed": int(passed),
        "failed": int(total - passed),
        "pass_rate_%": round(100 * passed / total, 1) if total else 0,
    }

check_df = pd.DataFrame(check_rows).T
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

check_df[["passed", "failed"]].plot(kind="bar", ax=ax1, color=["#70ad47", "#e07b39"], rot=25)
ax1.set_title("Check Results")
ax1.set_ylabel("Count")

ax2.bar(range(len(check_df)), check_df["pass_rate_%"], color="steelblue")
ax2.set_xticks(range(len(check_df)))
ax2.set_xticklabels(check_df.index, rotation=25, ha="right", fontsize=8)
ax2.axhline(98, color="crimson", linestyle="--", linewidth=1.2, label="98% threshold")
ax2.set_ylim(0, 102)
ax2.set_ylabel("Pass Rate (%)")
ax2.set_title("Check Pass Rate")
ax2.legend(fontsize=8)
for i, v in enumerate(check_df["pass_rate_%"]):
    ax2.text(i, v + 0.5, f"{v:.1f}%", ha="center", fontsize=9)

plt.tight_layout()
plt.show()